# 10 — The contest starter kit as a baseline, with its bugs fixed  *(branch `exp/10-starter-kit`)*

The FinAI Contest 2025 Task 1 starter kit (`experiments/starter_kit/`, MIT, SecureFinAI Lab), run on the contest's
**1-second** order-book data for 2021-04-07 → 04-13, **price only** (no DeepSeek signals): Alpha101 → the kit's
LSTM+GRU factor model → an ensemble of 4 DQN agents (D3QN, Double DQN ×2, Twin D3QN) with a majority vote.
70 / 15 / 15 chronological split; members selected on validation; the test split scored once, on every
consecutive 79-minute episode.

As shipped the kit never trades (`break_step = 32`). The fixes, each marked `[fix]` in the code
(`git diff 428db91 -- experiments/starter_kit` shows all of them):

| | as shipped | fixed |
|---|---|---|
| training budget | `break_step = 32` ("TODO reset to 32e4") | 320,000 steps |
| factor ↔ price alignment | factors 900 rows short, aligned to the rear: the agent saw a 15-min-old forecast | row t's factors next to row t's price |
| ensemble members | one seed for all; DoubleDQN ×2 and TwinD3QN identical; 2nd DoubleDQN overwritten on save | seed + index, own folder each, all 4 used |
| exploration | with the target network (τ = 2e-6) and its default ε = 0.125 | the online network, ε = 0.005 |
| return estimate for value normalisation | argmax *index* of Q | max Q |
| checkpoint | scored on 3 random training windows; best discarded, final weights kept | best mean return over all validation episodes |
| stop-loss | absolute $0.001 (train) / $0.0001 (eval) on a $60k price | 0.1 % of price, same in train and eval |
| kit bookkeeping | priced at `price_ary[step_i]` (episode counter) | the traded row |
| evaluation | 1 random 79-min window (of the training file) | every 79-min episode of the test split |
| RNN | "validation" loss on training rows; last network kept | held-out rows; best network |
| factor scaling | each file normalised on itself (test on its own future) | train-split quantiles for all rows |

Kept as shipped: slippage 7e-7, 1 BTC position on $1M, learning rate 2e-6, γ 0.995, network (128,128,128),
4096 parallel envs (halved automatically when the replay buffer — ~24 GB at 4096 — does not fit the GPU;
**pick an A100** to run it exactly as shipped).

**Inputs** (Drive, never committed — the contest data is confidential), either
* `thesis_rl/starter_kit/prepared/` with the prepared arrays (made locally with
  `python kit_pipeline.py prepare --csv BTC_1sec.csv --data prepared`; then the next cell skips the CPU-heavy part), or
* `thesis_rl/contest/data/BTC_1sec.csv` (or the slim `BTC_1sec_kit.csv`): Alpha101 then runs here, ~3–4 h on CPU.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/10-starter-kit"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > A100 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "10_starter_kit")
sys.path.insert(0, f"{EXP}/starter_kit")             # the kit's flat modules (erl_agent, trade_simulator, ...)
import kit_pipeline as kp

## Data: Alpha101, labels, splits (CPU) and the kit's RNN factor model (GPU)

In [ ]:
from pathlib import Path
KIT = Path(DRIVE_ROOT) / "starter_kit"
DATA = KIT / "prepared"
DATA.mkdir(parents=True, exist_ok=True)
if not (DATA / "BTC_1sec_input.npy").exists():
    csvs = [Path(DRIVE_ROOT) / "contest" / "data" / n for n in ("BTC_1sec_kit.csv", "BTC_1sec.csv")]
    csv = next((c for c in csvs if c.exists()), None)
    assert csv, f"upload the prepared arrays to {DATA} or the contest csv to {csvs[1].parent}"
    if not (DATA / "alpha101_raw.npy").exists():
        print("computing Alpha101 over 494,749 rows on the CPU - about 3-4 hours; cached on Drive when done")
    kp.prepare(str(csv), str(DATA))
print(open(DATA / "splits.json").read())
kp.rnn(str(DATA), gpu_id=0)                          # ~10 min on a GPU; skipped when the factors exist

## Ensemble: 4 members × 320,000 steps

Each member is saved when it finishes, so after a disconnect re-run this cell: finished members are skipped
(an unfinished member restarts). Roughly 20–40 min per member on an A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
RUN = KIT / "runs" / "shipped"                       # slippage 7e-7 as shipped
kp.ensemble(str(DATA), str(RUN), gpu_id=0, break_step=int(32e4), num_sims="auto", slippage=7e-7, seed=0)

## Results: validation (selection) and test (scored once)

In [ ]:
import json
import pandas as pd
pd.set_option("display.width", 200)
for m in sorted((RUN / "ensemble_models").iterdir()):
    s = json.loads((m / "selection.json").read_text())
    print(f"{m.name:18s} seed {s['seed']}  best validation return ${s['best_val_return']:8.2f} per episode at step {s['best_step']:,}")
val = pd.DataFrame(kp.evaluate(str(DATA), str(RUN), split="val", gpu_id=0))
test = pd.DataFrame(kp.evaluate(str(DATA), str(RUN), split="test", gpu_id=0))
cols = ["curve", "cum_return_pct", "max_drawdown_pct", "sharpe_per_step", "trades", "episodes",
        "episodes_positive", "episode_return_pct_median"]
print("\nVALIDATION"); display(val[cols])
print("TEST (held out)"); display(test[cols])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, lab in (("sim", "ensemble (simulator account)"), ("bh_1btc", "buy & hold 1 BTC"), ("kit", "kit bookkeeping")):
    e = np.load(RUN / "eval_test" / f"equity_{name}.npy")
    axes[0].plot((e / 1e6 - 1) * 100, lw=1.3, label=lab)
axes[0].set(title="Test: chained 79-min episodes, $1M", xlabel="2-second steps", ylabel="cumulative return (%)")
axes[0].axhline(0, color="grey", lw=0.8); axes[0].legend()
for m in sorted((RUN / "ensemble_models").iterdir()):
    c = np.array(json.loads((m / "selection.json").read_text())["val_curve"])
    axes[1].plot(c[:, 0], c[:, 1], lw=1.3, label=m.name)
axes[1].set(title="Validation return per episode during training", xlabel="training step", ylabel="$ per episode")
axes[1].axhline(0, color="grey", lw=0.8); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(RUN / "10_starter_kit.png", dpi=150); plt.show()

## Optional: the same at realistic costs

Slippage 7e-7 is ~0 (the thesis uses 0.12 % per trade). Set `RUN_REALISTIC = True` to train and score a
second ensemble with `slippage = 1.2e-3`; it takes as long as the first.

In [ ]:
RUN_REALISTIC = False
if RUN_REALISTIC:
    RUN_R = KIT / "runs" / "slippage_0.12pct"
    kp.ensemble(str(DATA), str(RUN_R), gpu_id=0, break_step=int(32e4), num_sims="auto", slippage=1.2e-3, seed=0)
    display(pd.DataFrame(kp.evaluate(str(DATA), str(RUN_R), split="test", gpu_id=0))[cols])